# GAN for SQLi — Vòng tinh chỉnh (targeted refinement) — SeqGAN Improved — File 1/7 — GPU

Notebook này chạy **hoàn toàn độc lập** (giống văn phong `phase3_oom_retry_5_of_6_STANDALONE.ipynb`), chỉ dựa vào project đã copy sẵn trong Drive tại `GAN_for_SQLi - refinement_improved/` — **không** đụng tới `results_seqgan_phase2b_phase3/` hay bất kỳ kết quả Phase 2A/2B/3 chính thức nào đã chạy trước đó.

**Đây KHÔNG PHẢI Phase 3 chính thức lần hai.** Phase 3 chính thức (64 run, luật "variant phải hợp lệ trên toàn bộ 8 dataset") đã đóng. Đây là **vòng tinh chỉnh bổ sung (ablation/refinement)** theo đúng phương pháp luận trong `thuyet_minh_lua_chon_D_V8.md`: khảo sát độ nhạy của các cấu hình đã biết là mạnh (anchor V8D + local champion từng family) khi đổi ratio sang `1:100`, `1:200`, `1:500`.

## Phạm vi file này

- Chỉ chạy **SeqGAN Improved** (method `seqgan_improved`).
- **33 run** = 11 tổ hợp (family, scenario, variant) × 3 ratio (100/200/500), chạy tuần tự trong cùng 1 session.
- 8 cell (family, scenario) dùng cho vòng tinh chỉnh (khác 6-scenario Borda gốc của Phase 2A):

| Family | Scenario | Vai trò |
|---|---|---|
| boolean | D | Anchor (V8D) |
| boolean | E | Local champion (V2) |
| error | D | Anchor (V8D) |
| error | B | Local champion = Anchor (V8) |
| time | A | Anchor = Local champion (V8) |
| time | D | Anchor = Local champion (V8) |
| union | D | Anchor (V8D) |
| union | F | Local champion (V3) |

- 11 dòng (family, scenario, variant) đúng theo bảng mục 6 của `thuyet_minh_lua_chon_D_V8.md`, nhân với 3 ratio ở bước chạy.

## Lưu ý cần đưa vào luận văn

Toàn bộ 64 run gốc của Phase 3 đã dừng adversarial sớm do `vanishing_reward` (chỉ 3–7/60–200 epoch). Nếu hiện tượng này lặp lại rõ hơn ở ratio cực lệch (đặc biệt `1:500`), cần bàn về ảnh hưởng của ratio lên tốc độ hội tụ reward, không chỉ xem là lỗi cấu hình — xem mục 7 của `thuyet_minh_lua_chon_D_V8.md`.


## 1. Mount Drive

In [8]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Copy project — trỏ đúng vào `GAN_for_SQLi - refinement_improved` (project độc lập, không phải Drive gốc `GAN_for_SQLi`, không đụng kết quả Phase 2A/2B/3 chính thức)

In [9]:
import csv
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import psutil
import torch
import yaml

# Hang so duong dan Drive rieng cho vong tinh chinh nay -- doi neu ten folder khac.
# Folder nay phai da co san code + data/SQLiV3.csv.zip (copy tu GAN_for_SQLi goc truoc khi chay).
PATH_COLAB = "/content/drive/MyDrive/GAN/GAN_for_SQLi/"

DRIVE_PROJECT = Path(PATH_COLAB.rstrip("/"))
RESULTS_ROOT = Path(PATH_COLAB) / "result"
LOCAL_PROJECT = Path("/content/GAN_for_SQLi")

assert DRIVE_PROJECT.is_dir(), f"Khong tim thay folder Drive: {DRIVE_PROJECT}"
assert (DRIVE_PROJECT / "scripts/research_pipeline.py").is_file(), (
    f"Khong tim thay project tai {DRIVE_PROJECT}"
)

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    DRIVE_PROJECT,
    LOCAL_PROJECT,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        "__pycache__", "*.pyc", "result", "results", "results_smoke", "results_mini",
        "results_cpu20_phase1_2a", "results_seqgan_only_mini",
        "results_seqgan_phase2b_phase3",
    ),
)
os.chdir(LOCAL_PROJECT)

print("Project (local, rieng cho vong tinh chinh nay):", LOCAL_PROJECT)
print("Results (ghi vao Drive, doc lap voi Phase 2A/2B/3 chinh thuc):", RESULTS_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Can runtime GPU (A100/L4/T4)"


Project (local, rieng cho vong tinh chinh nay): /content/GAN_for_SQLi
Results (ghi vao Drive, doc lap voi Phase 2A/2B/3 chinh thuc): /content/drive/MyDrive/GAN/GAN_for_SQLi/result
GPU: NVIDIA L4


## 3. Cài dependency

In [10]:
%pip install -q -r requirements.txt


## 4. `top2_scenarios_per_family.csv` thủ công — 8 cell (family, scenario) của vòng tinh chỉnh

`read_selected_cells()` trong `common/ingestion.py` bắt buộc **mỗi family khai báo trong config** (`boolean, union, time, error`) phải có đúng 2 scenario. Đây **không phải** bảng Borda gốc của Phase 2A — đây là 8 cell được chọn theo nguyên tắc anchor (D) + local champion từng family, theo mục 2–4 của `thuyet_minh_lua_chon_D_V8.md`.

In [11]:
MANUAL_TOP2 = [
    {"family": "boolean", "scenario": "D", "rank": 1, "source": "targeted_refinement_V8D_anchor"},
    {"family": "boolean", "scenario": "E", "rank": 2, "source": "targeted_refinement_local_champion_V2"},
    {"family": "error",   "scenario": "D", "rank": 1, "source": "targeted_refinement_V8D_anchor"},
    {"family": "error",   "scenario": "B", "rank": 2, "source": "targeted_refinement_local_champion_V8"},
    {"family": "time",    "scenario": "A", "rank": 1, "source": "targeted_refinement_local_champion_V8"},
    {"family": "time",    "scenario": "D", "rank": 2, "source": "targeted_refinement_V8D_anchor"},
    {"family": "union",   "scenario": "D", "rank": 1, "source": "targeted_refinement_V8D_anchor"},
    {"family": "union",   "scenario": "F", "rank": 2, "source": "targeted_refinement_local_champion_V3"},
]

MANUAL_SELECTION_PATH = LOCAL_PROJECT / "data" / "manual_top2_refinement_improved.csv"
MANUAL_SELECTION_PATH.parent.mkdir(parents=True, exist_ok=True)
with MANUAL_SELECTION_PATH.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["family", "scenario", "rank", "source"])
    writer.writeheader()
    writer.writerows(MANUAL_TOP2)
print("Da ghi:", MANUAL_SELECTION_PATH)
pd.read_csv(MANUAL_SELECTION_PATH)


Da ghi: /content/GAN_for_SQLi/data/manual_top2_refinement_improved.csv


,family,scenario,rank,source
0,boolean,D,1,targeted_refinement_V8D_anchor
1,boolean,E,2,targeted_refinement_local_champion_V2
2,error,D,1,targeted_refinement_V8D_anchor
3,error,B,2,targeted_refinement_local_champion_V8
4,time,A,1,targeted_refinement_local_champion_V8
5,time,D,2,targeted_refinement_V8D_anchor
6,union,D,1,targeted_refinement_V8D_anchor
7,union,F,2,targeted_refinement_local_champion_V3


## 5. Cấu hình runtime — giữ nguyên hyperparameter gốc V1–V8 và `seqgan_improved` (không hạ chuẩn), `results_root` trỏ vào `PATH_COLAB/result`

Mặc định file này **không** giảm epoch/rollout — dùng đúng cấu hình đã khóa trong `configs/experiment_config.yaml` (adversarial 200 epoch, rollout 16, generator pretrain 120/160 theo variant) để kết quả vòng tinh chỉnh so sánh được trực tiếp với Phase 3 gốc. Nếu Colab báo OOM hoặc hết thời gian session, mở khối `TUY CHON HA CHUAN` bên dưới và bỏ comment.

In [12]:
base_config = yaml.safe_load(
    (LOCAL_PROJECT / "configs/experiment_config.yaml").read_text(encoding="utf-8")
)
config = json.loads(json.dumps(base_config))

# --- TUY CHON HA CHUAN (mac dinh TAT) -- chi bo comment neu gap OOM / qua thoi gian session ---
# for variant in config["phase3"]["variants"]:
#     variant["generator_pretrain_epochs"] = int(variant["generator_pretrain_epochs"] // 2)
# config["generation"]["seqgan_improved"].update({
#     "adversarial_epochs": 100,
#     "rollout_count": 8,
# })
# config["generation"]["n_samples"] = 1000

# --- Diem quan trong: results_root tro vao PATH_COLAB/result, doc lap hoan toan voi Phase 2A/2B/3 chinh thuc ---
config["outputs"]["results_root"] = str(RESULTS_ROOT)

RUNTIME_CONFIG_DIR = Path("/content/colab_configs")
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_CONFIG_DIR / "experiment_refinement_improved.yaml"
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

PIPELINE = [sys.executable, "scripts/research_pipeline.py", "--config", str(RUNTIME_CONFIG)]

print("Runtime config:", RUNTIME_CONFIG)
print("results_root ->", config["outputs"]["results_root"])


Runtime config: /content/colab_configs/experiment_refinement_improved.yaml
results_root -> /content/drive/MyDrive/GAN/GAN_for_SQLi/result


## 6. Prepare data (đọc `data/SQLiV3.csv.zip` có sẵn trong project độc lập)

In [13]:
prepare_result = subprocess.run(
    [*PIPELINE, "prepare-data"], cwd=LOCAL_PROJECT, capture_output=True, text=True, check=False,
)
print(prepare_result.stdout)
if prepare_result.stderr:
    print(prepare_result.stderr)
if prepare_result.returncode != 0:
    raise RuntimeError(f"prepare-data that bai: {prepare_result.returncode}")
print("prepare-data completed")


Prepared fixed data with seed 88
Phase 2A target=312 feasible=True

prepare-data completed


## 7. `prepare-phase2b` một lần cho cả 8 cell — sinh `attack_train.csv` cho toàn bộ 7 ratio trong grid (`full, 10, 20, 50, 100, 200, 500`), chưa freeze ratio nào

In [14]:
prepare2b_result = subprocess.run(
    [*PIPELINE, "prepare-phase2b", "--selection", str(MANUAL_SELECTION_PATH)],
    cwd=LOCAL_PROJECT, capture_output=True, text=True, check=False,
)
print(prepare2b_result.stdout)
if prepare2b_result.stderr:
    print(prepare2b_result.stderr)
if prepare2b_result.returncode != 0:
    raise RuntimeError(f"prepare-phase2b that bai: {prepare2b_result.returncode}")

PREFLIGHT_2B_PATH = LOCAL_PROJECT / "data" / "prepared" / "phase2b" / "preflight.csv"
assert PREFLIGHT_2B_PATH.exists(), f"Khong tao duoc preflight: {PREFLIGHT_2B_PATH}"
preflight = pd.read_csv(PREFLIGHT_2B_PATH)
TARGET_RATIOS = ["100", "200", "500"]
check = preflight[preflight["ratio"].astype(str).isin(TARGET_RATIOS)]
print(check[["family", "scenario", "ratio", "target", "capacity", "status"]].to_string(index=False))
assert (check["status"] == "ready").all(), (
    "Co cell/ratio insufficient_pool -- xem preflight.csv truoc khi chay tiep."
)


Phase 2B datasets: 49/56 ready

 family scenario ratio  target  capacity status
boolean        D   100     156      1842  ready
boolean        D   200      78      1842  ready
boolean        D   500      31      1842  ready
boolean        E   100     156      3669  ready
boolean        E   200      78      3669  ready
boolean        E   500      31      3669  ready
  error        D   100     156       522  ready
  error        D   200      78       522  ready
  error        D   500      31       522  ready
  error        B   100     156      1036  ready
  error        B   200      78      1036  ready
  error        B   500      31      1036  ready
   time        A   100     156      1186  ready
   time        A   200      78      1186  ready
   time        A   500      31      1186  ready
   time        D   100     156       597  ready
   time        D   200      78       597  ready
   time        D   500      31       597  ready
  union        D   100     156       864  ready
  union 

## 8. 11 tổ hợp (family, scenario, variant) cố định — đúng bảng mục 6 của `thuyet_minh_lua_chon_D_V8.md`

Ratio được nhân ở bước chạy (mục 10), không lặp lại ở đây.

In [15]:
# (family, scenario, variant, vai_tro) -- 11 dong, khop 1-1 voi bang muc 6 thuyet_minh_lua_chon_D_V8.md
REFINEMENT_ROWS = [
    ("boolean", "D", "V8", "Anchor"),
    ("boolean", "D", "V7", "Local champion (boolean/D)"),
    ("boolean", "E", "V2", "Local champion (boolean/E)"),
    ("error",   "D", "V4", "Local champion (error/D)"),
    ("error",   "B", "V8", "Local champion = Anchor (error/B)"),
    ("error",   "D", "V8", "Anchor"),
    ("time",    "A", "V8", "Anchor = Local champion"),
    ("time",    "D", "V8", "Anchor = Local champion"),
    ("union",   "D", "V8", "Anchor"),
    ("union",   "D", "V2", "Local champion (union/D)"),
    ("union",   "F", "V3", "Local champion (union/F)"),
]
assert len(REFINEMENT_ROWS) == 11
ROLE_LOOKUP = {(f, s, v): role for f, s, v, role in REFINEMENT_ROWS}
TARGET_TRIPLES = set(ROLE_LOOKUP.keys())
TARGET_RATIOS = ["100", "200", "500"]
print(f"{len(REFINEMENT_ROWS)} to hop (family,scenario,variant) x {len(TARGET_RATIOS)} ratio = "
      f"{len(REFINEMENT_ROWS) * len(TARGET_RATIOS)} run")


11 to hop (family,scenario,variant) x 3 ratio = 33 run


## 9. Vòng lặp chính: với mỗi ratio trong {100, 200, 500} → freeze → sinh `phase3/run_matrix.csv` → lọc đúng 11 dòng → chạy tuần tự (không dùng scheduler song song, tránh OOM)

`freeze-phase3` reset `data/prepared/frozen/` mỗi lần gọi, nhưng `run_id`/`out_dir` đã nhúng sẵn `R{ratio}` nên kết quả của ratio trước **không bị ghi đè** khi chuyển sang ratio sau — mỗi ratio ghi vào một thư mục riêng dưới `results_root/phase3/seqgan_improved/<family>/<scenario>/R<ratio>/<variant>/`.

In [ ]:
# ALL_RESULTS = []
# shard_dir = Path("/content/refinement_improved_shards")
# shard_dir.mkdir(parents=True, exist_ok=True)

# for ratio in TARGET_RATIOS:
#     print("#" * 70)
#     print(f"RATIO 1:{ratio}")
#     print("#" * 70)

#     # --- freeze dung ratio nay cho 8 cell ---
#     SELECTED_RATIO_PATH = RESULTS_ROOT / "phase2b" / f"selected_ratio_R{ratio}.json"
#     SELECTED_RATIO_PATH.parent.mkdir(parents=True, exist_ok=True)
#     SELECTED_RATIO_PATH.write_text(
#         json.dumps({
#             "selected_global_ratio": ratio,
#             "selection_method": "targeted_refinement_manual",
#             "note": f"Vong tinh chinh sau Phase 3 -- ratio 1:{ratio}, xem thuyet_minh_lua_chon_D_V8.md",
#         }, ensure_ascii=False, indent=2),
#         encoding="utf-8",
#     )
#     freeze_result = subprocess.run(
#         [*PIPELINE, "freeze-phase3", "--selection", str(MANUAL_SELECTION_PATH),
#          "--ratio", str(SELECTED_RATIO_PATH)],
#         cwd=LOCAL_PROJECT, capture_output=True, text=True, check=False,
#     )
#     print(freeze_result.stdout)
#     if freeze_result.stderr:
#         print(freeze_result.stderr)
#     if freeze_result.returncode != 0:
#         raise RuntimeError(f"freeze-phase3 that bai o ratio {ratio}: {freeze_result.returncode}")

#     # --- sinh lai phase3/run_matrix.csv (64 dong: 8 cell x 8 variant) roi loc dung 11 dong ---
#     matrix_result = subprocess.run(
#         [*PIPELINE, "matrix", "--phase", "phase3"],
#         cwd=LOCAL_PROJECT, capture_output=True, text=True, check=False,
#     )
#     print(matrix_result.stdout)
#     if matrix_result.stderr:
#         print(matrix_result.stderr)
#     if matrix_result.returncode != 0:
#         raise RuntimeError(f"Tao Phase 3 matrix that bai o ratio {ratio}: {matrix_result.returncode}")

#     PHASE3_MATRIX_PATH = RESULTS_ROOT / "phase3" / "run_matrix.csv"
#     assert PHASE3_MATRIX_PATH.exists(), f"Khong tim thay {PHASE3_MATRIX_PATH}"
#     full_matrix = pd.read_csv(PHASE3_MATRIX_PATH, dtype=str, keep_default_na=False)

#     my_rows = full_matrix.loc[
#         full_matrix.apply(
#             lambda r: (r["family"], r["scenario"], r["variant_id"]) in TARGET_TRIPLES, axis=1
#         )
#     ].copy()
#     assert len(my_rows) == 11, f"Ratio {ratio}: loc duoc {len(my_rows)} dong, ky vong 11"
#     assert (my_rows["data_status"] == "ready").all(), (
#         f"Ratio {ratio}: co dong data_status != ready -- kiem tra lai buoc freeze o tren."
#     )
#     print(f"Se chay {len(my_rows)} model o ratio 1:{ratio}:")
#     print(my_rows[["run_id", "family", "scenario", "ratio", "variant_id", "data_status"]].to_string(index=False))

#     # --- chay tuan tu tung dong, khong dung ca notebook neu 1 run loi (giu ket qua cac run khac) ---
#     for _, row in my_rows.iterrows():
#         run_id = row["run_id"]
#         role = ROLE_LOOKUP[(row["family"], row["scenario"], row["variant_id"])]
#         print("=" * 70)
#         print("Dang chay:", run_id, "| vai tro:", role)
#         print("=" * 70)
#         one_row_path = shard_dir / f"{run_id}.csv"
#         my_rows.loc[my_rows["run_id"].eq(run_id)].to_csv(one_row_path, index=False)
#         result = subprocess.run(
#             [*PIPELINE, "run-matrix", "--matrix", str(one_row_path),
#              "--steps", "all", "--execute", "--resume"],
#             cwd=LOCAL_PROJECT, check=False,
#         )
#         status = "OK" if result.returncode == 0 else f"LOI (return_code={result.returncode})"
#         if result.returncode != 0:
#             print(f"!!! {run_id} van loi. Xem log tai: {Path(row['out_dir']) / 'logs' / 'train.log'}")
#         else:
#             print(f">>> {run_id} OK")
#         ALL_RESULTS.append({
#             "run_id": run_id, "family": row["family"], "scenario": row["scenario"],
#             "ratio": ratio, "variant_id": row["variant_id"], "role": role,
#             "returncode": result.returncode, "status": status, "out_dir": row["out_dir"],
#         })

# print("#" * 70)
# print(f"HOAN TAT {len(ALL_RESULTS)}/33 run")
# print("#" * 70)


######################################################################
RATIO 1:100
######################################################################
Frozen 8 datasets at ratio 100

Wrote 64 rows to /content/drive/MyDrive/GAN/GAN_for_SQLi/result/phase3/run_matrix.csv
{"ready": 64}

Se chay 11 model o ratio 1:100:
                                       run_id  family scenario ratio variant_id data_status
phase3__seqgan_improved__boolean__D__R100__V7 boolean        D   100         V7       ready
phase3__seqgan_improved__boolean__D__R100__V8 boolean        D   100         V8       ready
phase3__seqgan_improved__boolean__E__R100__V2 boolean        E   100         V2       ready
  phase3__seqgan_improved__error__D__R100__V4   error        D   100         V4       ready
  phase3__seqgan_improved__error__D__R100__V8   error        D   100         V8       ready
  phase3__seqgan_improved__error__B__R100__V8   error        B   100         V8       ready
   phase3__seqgan_improved__time__A__

KeyboardInterrupt: 

In [20]:
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading, time
from datetime import datetime, timezone

# --- Buoc 1: tao 1 ban copy project rieng cho MOI ratio, tu LOCAL_PROJECT da prepare-data + prepare-phase2b ---
RATIO_PROJECTS = {}
for ratio in TARGET_RATIOS:
    ratio_project = Path(f"/content/GAN_for_SQLi_refinement_improved_R{ratio}")
    if ratio_project.exists():
        shutil.rmtree(ratio_project)
    shutil.copytree(LOCAL_PROJECT, ratio_project)  # copy sau khi da co du data/prepared/phase2b cho ca 7 ratio
    RATIO_PROJECTS[ratio] = ratio_project
    print("Da tao ban copy rieng cho ratio", ratio, "->", ratio_project)

# --- Buoc 2: freeze + sinh matrix RIENG cho tung ratio, khong dung chung frozen/ hay run_matrix.csv ---
ALL_ROWS = []  # gom du 33 dong, moi dong nho them project copy tuong ung

for ratio in TARGET_RATIOS:
    proj = RATIO_PROJECTS[ratio]

    SELECTED_RATIO_PATH = proj / "selected_ratio.json"
    SELECTED_RATIO_PATH.write_text(
        json.dumps({"selected_global_ratio": ratio, "selection_method": "targeted_refinement_manual"},
                    ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    freeze_result = subprocess.run(
        [*PIPELINE, "freeze-phase3", "--selection", str(MANUAL_SELECTION_PATH),
         "--ratio", str(SELECTED_RATIO_PATH)],
        cwd=proj, capture_output=True, text=True, check=False,
    )
    if freeze_result.returncode != 0:
        raise RuntimeError(f"freeze-phase3 that bai o ratio {ratio}: {freeze_result.stderr}")

    # --out tro ve file CUC BO trong tung ban copy -- khong dung chung 1 file tren Drive
    LOCAL_MATRIX_PATH = proj / f"local_run_matrix_R{ratio}.csv"
    matrix_result = subprocess.run(
        [*PIPELINE, "matrix", "--phase", "phase3", "--out", str(LOCAL_MATRIX_PATH)],
        cwd=proj, capture_output=True, text=True, check=False,
    )
    if matrix_result.returncode != 0:
        raise RuntimeError(f"Tao matrix that bai o ratio {ratio}: {matrix_result.stderr}")

    full_matrix = pd.read_csv(LOCAL_MATRIX_PATH, dtype=str, keep_default_na=False)
    my_rows = full_matrix.loc[
        full_matrix.apply(lambda r: (r["family"], r["scenario"], r["variant_id"]) in TARGET_TRIPLES, axis=1)
    ].copy()
    assert len(my_rows) == 11, f"Ratio {ratio}: loc duoc {len(my_rows)} dong, ky vong 11"
    assert (my_rows["data_status"] == "ready").all()

    for _, row in my_rows.iterrows():
        row_dict = row.to_dict()
        row_dict["_project"] = proj  # nho project copy de tro cwd dung khi chay
        ALL_ROWS.append(row_dict)

print(f"Tong so dong se chay dong thoi: {len(ALL_ROWS)} (ky vong 33)")

# --- Buoc 3: chay dong thoi CA 33, moi row dung dung project copy cua no ---
shard_dir = Path("/content/refinement_improved_shards_full")
shard_dir.mkdir(parents=True, exist_ok=True)

def run_one_row_full(row: dict) -> dict:
    run_id = row["run_id"]
    proj = row["_project"]
    role = ROLE_LOOKUP[(row["family"], row["scenario"], row["variant_id"])]
    one_row_path = shard_dir / f"{run_id}.csv"
    pd.DataFrame([{k: v for k, v in row.items() if k != "_project"}]).to_csv(one_row_path, index=False)

    result = subprocess.run(
        [*PIPELINE, "run-matrix", "--matrix", str(one_row_path), "--steps", "all", "--execute", "--resume"],
        cwd=proj, check=False,
    )
    return {
        "run_id": run_id, "family": row["family"], "scenario": row["scenario"],
        "ratio": row["ratio"], "variant_id": row["variant_id"], "role": role,
        "returncode": result.returncode,
        "status": "OK" if result.returncode == 0 else f"LOI (return_code={result.returncode})",
        "out_dir": row["out_dir"],
    }

active_runs = {row["run_id"]: Path(row["out_dir"]) for row in ALL_ROWS}
stop_event = threading.Event()
monitor_thread = threading.Thread(target=monitor_progress, args=(active_runs, stop_event, 15), daemon=True)
monitor_thread.start()

ALL_RESULTS = []
try:
    with ThreadPoolExecutor(max_workers=len(ALL_ROWS)) as executor:  # 33 -- xem canh bao ben duoi
        future_map = {executor.submit(run_one_row_full, row): row["run_id"] for row in ALL_ROWS}
        for i, future in enumerate(as_completed(future_map), start=1):
            run_id = future_map[future]
            try:
                result = future.result()
            except Exception as error:
                result = {"run_id": run_id, "returncode": -1, "status": f"EXCEPTION: {error!r}"}
            active_runs.pop(run_id, None)
            print(f"[{i}/{len(ALL_ROWS)}] {result['status']} | {run_id}")
            ALL_RESULTS.append(result)
finally:
    stop_event.set()
    monitor_thread.join(timeout=2)

print(f"HOAN TAT {len(ALL_RESULTS)}/33 run")

Da tao ban copy rieng cho ratio 100 -> /content/GAN_for_SQLi_refinement_improved_R100
Da tao ban copy rieng cho ratio 200 -> /content/GAN_for_SQLi_refinement_improved_R200
Da tao ban copy rieng cho ratio 500 -> /content/GAN_for_SQLi_refinement_improved_R500
Tong so dong se chay dong thoi: 33 (ky vong 33)
[1/33] OK | phase3__seqgan_improved__boolean__D__R100__V7

--- [03:19:00] Dang chay song song 32 run ---
  phase3__seqgan_improved__boolean__D__R100__V8: (loi doc log: IsADirectoryError(21, 'Is a directory'))
  phase3__seqgan_improved__boolean__E__R100__V2: (loi doc log: IsADirectoryError(21, 'Is a directory'))
  phase3__seqgan_improved__error__D__R100__V4: (loi doc log: IsADirectoryError(21, 'Is a directory'))
  phase3__seqgan_improved__error__D__R100__V8: (loi doc log: IsADirectoryError(21, 'Is a directory'))
  phase3__seqgan_improved__error__B__R100__V8: (loi doc log: IsADirectoryError(21, 'Is a directory'))
  phase3__seqgan_improved__time__A__R100__V8: (loi doc log: IsADirectoryErr

: 

## 10. Tổng hợp kết quả 33 run — đọc `run_manifest.json` từng run, lưu bảng tóm tắt vào `result/phase3/refinement_run_summary.csv`

In [ ]:
summary_rows = []
for entry in ALL_RESULTS:
    manifest_path = Path(entry["out_dir"]) / "run_manifest.json"
    manifest_status = "khong_tim_thay_manifest"
    stop_reason = ""
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest_status = manifest.get("status", "unknown")
        stop_reason = manifest.get("stop_reason", "")
    summary_rows.append({**entry, "manifest_status": manifest_status, "stop_reason": stop_reason})

summary_df = pd.DataFrame(summary_rows)
print(summary_df[["run_id", "family", "scenario", "ratio", "variant_id", "role",
                   "status", "manifest_status"]].to_string(index=False))

SUMMARY_PATH = RESULTS_ROOT / "phase3" / "refinement_run_summary.csv"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(SUMMARY_PATH, index=False)
print()
print("Da ghi bang tong hop:", SUMMARY_PATH)
print()
print("So run OK:", (summary_df["returncode"] == 0).sum(), "/", len(summary_df))
